# PS S6E4 - exp_20260409_026_realmlp_pair_te_min

目的:
 - Mahog_RealMLP_lb0.97685 の骨格から、最小再現版を作る
 - 018 / 024 / 025 と異なる人格候補として、RealMLP + pairwise TE を確認する

今回残すもの:
 - 元コードで使われている selected columns
 - pair combinations
 - multiclass TargetEncoder
 - RealMLP 本体

今回切るもの:
 - 余計な周辺最適化
 - 実験を重くするだけの拡張

## Imports

In [1]:
import os
import gc
import json
import math
import random
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore", category=pd.errors.ChainedAssignmentError)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260409_026_realmlp_pair_te_min"
    EXP_NAME = "realmlp_pair_te_min"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    TARGET = "Irrigation_Need"
    ID_COL = "id"
    SEED = 42
    N_FOLDS = 5

    NUMS = [
        "Soil_pH",
        "Soil_Moisture",
        "Organic_Carbon",
        "Electrical_Conductivity",
        "Temperature_C",
        "Humidity",
        "Rainfall_mm",
        "Sunlight_Hours",
        "Wind_Speed_kmh",
        "Field_Area_hectare",
        "Previous_Irrigation_mm",
    ]

    CATS = [
        "Soil_Type",
        "Crop_Type",
        "Crop_Growth_Stage",
        "Season",
        "Irrigation_Type",
        "Water_Source",
        "Mulching_Used",
        "Region",
    ]

    # shared code 準拠で pair 候補を作る元列
    PAIR_SOURCE_COLUMNS = [
        "Soil_Moisture",
        "Crop_Growth_Stage",
        "Temperature_C",
        "Mulching_Used",
        "Wind_Speed_kmh",
        "Rainfall_mm",
    ]

    PAIR_R = 2

    MODEL_CONFIG = {
        "n_ens": 8,
        "embed_dim": 6,
        "onehot_thresh": 8,
        "hidden_dims": [512, 64, 128],
        "dropout": 0.05,
        "activation": nn.SiLU,
        "add_front_scale": True,

        "pbld_hidden_dim": 20,
        "pbld_out_dim": 5,
        "pbld_freq_scale": 1.0,
        "pbld_activation": nn.GELU,
        "pbld_lr_factor": 0.1,

        "lr": 1e-2,
        "mom": 0.9,
        "sq_mom": 0.98,
        "lr_sched": "flat_cos",
        "flat_ratio": 0.3,
        "first_layer_lr_factor": 1.0,
        "lr_scale_mult": 10.0,
        "lr_bias_mult": 0.1,
        "weight_decay": 1e-2,
        "wd_scale_mult": 0.1,
        "wd_bias_mult": 0.5,
        "grad_clip": 1.0,

        "ls_eps": 0.04,
        "ls_eps_sched": "cos",
        "p_drop_sched": "expm4t",

        "use_early_stopping": False,
        "early_stopping_additive_patience": 20,
        "early_stopping_multiplicative_patience": 2.0,

        "epochs": 3,
        "train_bs": 256,
        "eval_bs": 10240,
        "verbosity": 2,

        "tfms": ["median_center", "robust_scale", "smooth_clip"],
        "device": "cuda",
        "random_state": 42,
    }

## utility

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def metric(y_true, y_pred):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)

def seed_everything(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)

## load data

In [4]:
train = pd.read_csv(CFG.TRAIN_PATH, index_col=CFG.ID_COL)
test = pd.read_csv(CFG.TEST_PATH, index_col=CFG.ID_COL)
orig = pd.read_csv(CFG.ORIG_PATH)

train[CFG.TARGET] = train[CFG.TARGET].map({"Low": 0, "Medium": 1, "High": 2})
orig[CFG.TARGET] = orig[CFG.TARGET].map({"Low": 0, "Medium": 1, "High": 2})

print("train.shape =", train.shape)
print("test.shape  =", test.shape)
print("orig.shape  =", orig.shape)
display(train.head())

train.shape = (630000, 20)
test.shape  = (270000, 19)
orig.shape  = (10000, 20)


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,0
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,0
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,0
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,1
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,0


## preprocess based on the shared

In [5]:
for c in CFG.NUMS:
    train[f"{c}_2"] = train[c].copy()
    test[f"{c}_2"] = test[c].copy()
    orig[f"{c}_2"] = orig[c].copy()

combined = pd.concat([train, test, orig])

for c in CFG.CATS + CFG.NUMS:
    combined[c], _ = combined[c].factorize()

combined[CFG.CATS + CFG.NUMS] = combined[CFG.CATS + CFG.NUMS].astype("category")

train = combined[:len(train)].copy()
test = combined[len(train):len(train) + len(test)].drop(CFG.TARGET, axis=1).copy()
orig = combined[len(train) + len(test):].copy()

print(train.shape, test.shape, orig.shape)

(630000, 31) (270000, 30) (10000, 31)


## make pair columns

In [6]:
TE_columns = []

for cols in combinations(CFG.PAIR_SOURCE_COLUMNS, CFG.PAIR_R):
    name = "-".join(cols)

    train[name] = train[cols[0]].astype(str)
    for col in cols[1:]:
        train[name] = train[name] + "_" + train[col].astype(str)

    test[name] = test[cols[0]].astype(str)
    for col in cols[1:]:
        test[name] = test[name] + "_" + test[col].astype(str)

    orig[name] = orig[cols[0]].astype(str)
    for col in cols[1:]:
        orig[name] = orig[name] + "_" + orig[col].astype(str)

    combined_pair = pd.concat([train[name], test[name], orig[name]], ignore_index=True)
    combined_pair, _ = combined_pair.factorize()

    if pd.Series(combined_pair).nunique() > len(combined_pair) // 2:
        train = train.drop(name, axis=1)
        test = test.drop(name, axis=1)
        orig = orig.drop(name, axis=1)
        continue

    train[name] = combined_pair[:len(train)]
    test[name] = combined_pair[len(train):len(train) + len(test)]
    orig[name] = combined_pair[len(train) + len(test):]

    TE_columns.append(name)

FEATURES = train.columns.tolist()
FEATURES.remove(CFG.TARGET)

print("n_pair_columns =", len(TE_columns))
print("TE_columns =", TE_columns)
print("n_features_before_te =", len(FEATURES))

n_pair_columns = 9
TE_columns = ['Soil_Moisture-Crop_Growth_Stage', 'Soil_Moisture-Mulching_Used', 'Crop_Growth_Stage-Temperature_C', 'Crop_Growth_Stage-Mulching_Used', 'Crop_Growth_Stage-Wind_Speed_kmh', 'Crop_Growth_Stage-Rainfall_mm', 'Temperature_C-Mulching_Used', 'Mulching_Used-Wind_Speed_kmh', 'Mulching_Used-Rainfall_mm']
n_features_before_te = 39


## RealMLP components

In [7]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X: np.ndarray, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self

    def transform(self, X: np.ndarray, y=None) -> np.ndarray:
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X


class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []

        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                emb = nn.ModuleList([nn.Embedding(dim, embed_dim) for _ in range(n_ens)])
                self.embed_layers.append(emb)
                self._embed_feature_indices.append(i)

    def forward(self, x):
        batch_size, n_ens, _ = x.shape
        features = []

        if self.onehot_features:
            onehot_x = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_oh = sum(onehot_dims)
            encoded = torch.zeros(batch_size, n_ens, total_oh, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx:idx+1].long()
                encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(encoded)

        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = []
            for model_idx in range(self.n_ens):
                indices = x[:, model_idx, feat_idx:feat_idx+1].long()
                feat_embs.append(emb_list[model_idx](indices))
            feat_combined = torch.cat(feat_embs, dim=1)
            features.append(feat_combined)

        return torch.cat(features, dim=2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class NTPLinear(nn.Module):
    def __init__(self, n_ens, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class PBLDEmbedding(nn.Module):
    def __init__(self, n_ens, n_features, hidden_dim=16, out_dim=4, freq_scale=0.1, activation=nn.GELU):
        super().__init__()
        self.n_ens = n_ens
        self.n_features = n_features
        self.out_dim = out_dim
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(
            torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim)
        )
        self.b2 = nn.Parameter(torch.randn(n_ens, n_features, out_dim - 1))
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        periodic = torch.cos(
            2 * math.pi * (x.unsqueeze(-1) * self.w1.unsqueeze(0) + self.b1.unsqueeze(0))
        )
        transformed = self.act(
            torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2) + self.b2.unsqueeze(0)
        )
        feat = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        return feat.flatten(start_dim=2)


class RealMLP(nn.Module):
    def __init__(self, output_dim, cat_dims, n_numerical, cfg):
        super().__init__()
        n_ens = cfg["n_ens"]
        embed_dim = cfg["embed_dim"]
        self.n_ens = n_ens

        self.cate = CategoricalFeatureLayer(
            n_ens=n_ens,
            cat_dims=cat_dims,
            embed_dim=embed_dim,
            onehot_thresh=cfg["onehot_thresh"],
        )

        self.num_embed = PBLDEmbedding(
            n_ens=n_ens,
            n_features=n_numerical,
            hidden_dim=cfg["pbld_hidden_dim"],
            out_dim=cfg["pbld_out_dim"],
            freq_scale=cfg["pbld_freq_scale"],
            activation=cfg["pbld_activation"],
        )

        num_emb_dim = n_numerical * cfg["pbld_out_dim"]
        cat_emb_dim = sum(c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total_dim = num_emb_dim + cat_emb_dim
        hidden_dims = cfg["hidden_dims"]

        act = cfg["activation"]

        layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens=n_ens, n_features=total_dim))

        self._dropout_modules = []
        in_dim = total_dim
        for i, out_dim_h in enumerate(hidden_dims):
            linear = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=out_dim_h)
            if i == 0:
                self.first_linear = linear
            drop = nn.Dropout(cfg["dropout"])
            self._dropout_modules.append(drop)
            layers += [linear, act(), drop]
            in_dim = out_dim_h

        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        combined = torch.cat([x_num, x_cat], dim=2)
        x = self.hidden(combined)
        x = self.output_layer(x)
        return F.softmax(x, dim=2)

## schedule and wrapper

In [8]:
def apply_schedule(init_value, progress, sched, flat_ratio=0.3):
    if sched == "constant":
        return init_value
    elif sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    elif sched == "flat_cos":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (math.cos(math.pi * t) + 1) / 2
    elif sched == "flat_anneal":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - t)
    elif sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    elif sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    else:
        raise ValueError(f"Unknown schedule: '{sched}'")


def get_parameter_groups(model, p):
    first_linear_weight_id = id(model.first_linear.weight)

    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)

    LR = p["lr"]
    WD = p["weight_decay"]
    return [
        {"params": scale_p,   "lr": LR * p["lr_scale_mult"],         "weight_decay": WD * p["wd_scale_mult"], "group": "scale"},
        {"params": pbld_p,    "lr": LR * p["pbld_lr_factor"],        "weight_decay": WD,                       "group": "pbld"},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay": WD,                       "group": "first_w"},
        {"params": other_w_p, "lr": LR,                              "weight_decay": WD,                       "group": "other_w"},
        {"params": bias_p,    "lr": LR * p["lr_bias_mult"],          "weight_decay": WD * p["wd_bias_mult"],  "group": "bias"},
    ]


def smooth_ce_loss(y_true, y_pred, ls=0.0, class_weights=None):
    n_classes = y_pred.size(1)
    y_smooth = torch.full_like(y_pred, ls / n_classes)
    y_smooth.scatter_(1, y_true.unsqueeze(1), 1.0 - ls + ls / n_classes)
    per_sample_loss = -(y_smooth * torch.log(y_pred.clamp(1e-15, 1))).sum(dim=1)
    if class_weights is not None:
        sample_weights = class_weights[y_true]
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum()
    return per_sample_loss.mean()


class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kwargs):
        self.params = {**CFG.MODEL_CONFIG, **kwargs}

    def fit(self, X_train, y_train, X_val, y_val, cat_col_names=None, ckpt_path="realmlp_ckpt.pth"):
        p = self.params
        dev = torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        verbose = p["verbosity"]
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        X_tr_num = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr = np.asarray(y_train)
        y_v = np.asarray(y_val)

        self.preprocessor_ = NumericalPreprocessor(p["tfms"])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names

        if cat_col_names:
            all_cat = np.concatenate([X_tr_cat, X_val_cat], axis=0)
            cat_dims = (all_cat.max(axis=0) + 1).tolist()
        else:
            cat_dims = []

        self.cat_dims_ = cat_dims
        if cat_dims:
            cat_max = np.array(cat_dims) - 1
            X_tr_cat = np.clip(X_tr_cat, 0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        classes = np.unique(y_tr)
        self.classes_ = classes
        weights_np = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weights = torch.as_tensor(weights_np, dtype=torch.float32, device=dev)

        self.model_ = RealMLP(
            output_dim=len(classes),
            cat_dims=cat_dims,
            n_numerical=X_tr_num.shape[1],
            cfg=p,
        ).to(dev)

        param_groups = get_parameter_groups(self.model_, p)
        for g in param_groups:
            g["lr_base"] = g["lr"]

        optimizer = torch.optim.AdamW(param_groups, betas=(p["mom"], p["sq_mom"]))

        Xtn = torch.as_tensor(X_tr_num, dtype=torch.float32, device=dev)
        Xtc = torch.as_tensor(X_tr_cat, dtype=torch.long, device=dev)
        ytt = torch.as_tensor(y_tr, dtype=torch.long, device=dev)
        Xvn = torch.as_tensor(X_val_num, dtype=torch.float32, device=dev)
        Xvc = torch.as_tensor(X_val_cat, dtype=torch.long, device=dev)

        n_ens = p["n_ens"]
        train_bs = p["train_bs"]
        eval_bs = p["eval_bs"]
        epochs = p["epochs"]
        lr_sched = p["lr_sched"]
        flat_ratio = p["flat_ratio"]
        total_steps = epochs * len(y_tr)
        train_order = np.arange(len(y_tr))

        best_score = -np.inf
        best_epoch = 0
        best_val_probs = None
        self.ckpt_path_ = ckpt_path

        for epoch in range(epochs):
            self.model_.train()
            for start in range(0, len(y_tr), train_bs):
                progress = (epoch * len(y_tr) + start) / total_steps
                idx_batch = train_order[start:start + train_bs]

                for g in optimizer.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, lr_sched, flat_ratio)

                optimizer.zero_grad()
                y_pred = self.model_(Xtn[idx_batch], Xtc[idx_batch])

                ls_val = apply_schedule(p["ls_eps"], progress, p["ls_eps_sched"], flat_ratio)
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"], flat_ratio)
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val

                loss = smooth_ce_loss(
                    ytt[idx_batch].repeat_interleave(n_ens),
                    y_pred.reshape(-1, len(classes)),
                    ls=ls_val,
                    class_weights=class_weights,
                )

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                optimizer.step()

            np.random.shuffle(train_order)

            self.model_.eval()
            with torch.no_grad():
                val_probs = np.concatenate([
                    self.model_(Xvn[s:s + eval_bs], Xvc[s:s + eval_bs]).mean(dim=1).cpu().numpy()
                    for s in range(0, len(y_v), eval_bs)
                ], axis=0)

            epoch_score = balanced_accuracy_score(y_v, np.argmax(val_probs, axis=1))
            improved = epoch_score > best_score
            if improved:
                best_score = epoch_score
                best_epoch = epoch + 1
                best_val_probs = val_probs.copy()
                torch.save(self.model_.state_dict(), ckpt_path)

            if verbose >= 2:
                print(
                    f"  epoch {epoch + 1}/{epochs}  "
                    f"bal-acc = {epoch_score:.5f}  "
                    f"best = {best_score:.5f}"
                    + ("  ✓" if improved else "")
                )

        self.model_.load_state_dict(torch.load(ckpt_path))
        self.best_score_ = best_score
        self.best_val_probs_ = best_val_probs
        self._dev = dev
        return self

    def predict_proba(self, X):
        eval_bs = self.params["eval_bs"]
        X_num = self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        X_cat = X[self.cat_col_names_].values.astype(np.int64)
        X_cat = np.clip(X_cat, 0, np.array(self.cat_dims_) - 1)

        Xn = torch.as_tensor(X_num, dtype=torch.float32, device=self._dev)
        Xc = torch.as_tensor(X_cat, dtype=torch.long, device=self._dev)

        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([
                self.model_(Xn[s:s + eval_bs], Xc[s:s + eval_bs]).mean(dim=1).cpu().numpy()
                for s in range(0, len(X_num), eval_bs)
            ], axis=0)

## CV main

In [9]:
oof = np.zeros((len(train), 3))
pred = np.zeros((len(test), 3))

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, random_state=42, shuffle=True)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train)), train[CFG.TARGET]), start=1):
    
    print(f"Fold {fold} / {CFG.N_FOLDS}")
    X_train = train.iloc[train_idx][FEATURES].copy()
    X_val = train.iloc[val_idx][FEATURES].copy()
    y_train = train.iloc[train_idx][CFG.TARGET].copy()
    y_val = train.iloc[val_idx][CFG.TARGET].copy()
    X_test = test.copy()

    encoder = TargetEncoder(target_type="multiclass", cv=5, random_state=42)

    result = pd.DataFrame(encoder.fit_transform(X_train[TE_columns], y_train), index=X_train.index)
    X_train = pd.concat([result.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)

    result = pd.DataFrame(encoder.transform(X_val[TE_columns]), index=X_val.index)
    X_val = pd.concat([result.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)

    result = pd.DataFrame(encoder.transform(X_test[TE_columns]), index=X_test.index)
    X_test = pd.concat([result.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)

    X_train = X_train.drop(TE_columns, axis=1)
    X_val = X_val.drop(TE_columns, axis=1)
    X_test = X_test.drop(TE_columns, axis=1)

    nums = [c for c in X_train.columns.tolist() if c not in CFG.CATS + CFG.NUMS]
    cats = CFG.CATS + CFG.NUMS

    model = RealMLP_TD_Classifier(**CFG.MODEL_CONFIG)
    model.fit(
        X_train, y_train,
        X_val, y_val,
        cat_col_names=cats,
        ckpt_path=f"realmlp_fold{fold}.pth",
    )

    oof[val_idx] = model.best_val_probs_
    pred += model.predict_proba(X_test)

    fold_score = metric(y_val.values, oof[val_idx])
    fold_scores.append(float(fold_score))
    print(f"Fold {fold}: {fold_score:.6f}")

    del X_train, y_train, X_val, y_val, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

pred /= CFG.N_FOLDS
raw_cv_ba = metric(train[CFG.TARGET].values, oof)
print(f"raw_cv_ba = {raw_cv_ba:.7f}")

Fold 1 / 5
  epoch 1/3  bal-acc = 0.97477  best = 0.97477  ✓
  epoch 2/3  bal-acc = 0.97510  best = 0.97510  ✓
  epoch 3/3  bal-acc = 0.97667  best = 0.97667  ✓
Fold 1: 0.976666
Fold 2 / 5
  epoch 1/3  bal-acc = 0.97593  best = 0.97593  ✓
  epoch 2/3  bal-acc = 0.97744  best = 0.97744  ✓
  epoch 3/3  bal-acc = 0.97800  best = 0.97800  ✓
Fold 2: 0.978004
Fold 3 / 5
  epoch 1/3  bal-acc = 0.97678  best = 0.97678  ✓
  epoch 2/3  bal-acc = 0.97805  best = 0.97805  ✓
  epoch 3/3  bal-acc = 0.97982  best = 0.97982  ✓
Fold 3: 0.979822
Fold 4 / 5
  epoch 1/3  bal-acc = 0.97619  best = 0.97619  ✓
  epoch 2/3  bal-acc = 0.97735  best = 0.97735  ✓
  epoch 3/3  bal-acc = 0.97787  best = 0.97787  ✓
Fold 4: 0.977867
Fold 5 / 5
  epoch 1/3  bal-acc = 0.97597  best = 0.97597  ✓
  epoch 2/3  bal-acc = 0.97776  best = 0.97776  ✓
  epoch 3/3  bal-acc = 0.97765  best = 0.97776
Fold 5: 0.977756
raw_cv_ba = 0.9780230


## simple bias search

In [10]:
best_score = raw_cv_ba
best_class_weights = np.ones(3, dtype=float)

grid = [
    [1.0, 1.0, 1.0],
    [1.2, 1.2, 1.2],
    [1.5, 1.5, 1.5],
    [1.5, 1.3, 1.8],
    [1.7, 1.5, 2.3],
    [1.8, 1.5, 2.4],
]

for cw in grid:
    cw = np.array(cw, dtype=float)
    adjusted = oof * cw
    adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
    score = metric(train[CFG.TARGET].values, adjusted)
    if score > best_score:
        best_score = score
        best_class_weights = cw.copy()

print("best_class_weights =", best_class_weights.tolist())
print("biased_cv_ba =", best_score)
print("improvement =", best_score - raw_cv_ba)

best_class_weights = [1.7, 1.5, 2.3]
biased_cv_ba = 0.9781126952221663
improvement = 8.966168936774821e-05


## final prediction

In [11]:
final_test_probs = pred * best_class_weights
final_test_probs = final_test_probs / final_test_probs.sum(axis=1, keepdims=True)
final_test_preds = np.argmax(final_test_probs, axis=1)

submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")
submission[CFG.TARGET] = final_test_preds
submission[CFG.TARGET] = submission[CFG.TARGET].map({0: "Low", 1: "Medium", 2: "High"})
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", pred)

oof_biased = oof * best_class_weights
oof_biased = oof_biased / oof_biased.sum(axis=1, keepdims=True)

pred_biased = final_test_probs.copy()

np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba_biased.npy", oof_biased)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba_biased.npy", pred_biased)

## save metadata

In [12]:
with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "raw_cv_ba": float(raw_cv_ba),
            "biased_cv_ba": float(best_score),
            "fold_scores": fold_scores,
            "best_class_weights": best_class_weights.tolist(),
            "n_pair_columns": len(TE_columns),
            "pair_columns": TE_columns,
            "n_features_before_te": len(FEATURES),
            "n_features_after_te": len(X_test.columns),
            "model_config_epochs": CFG.MODEL_CONFIG["epochs"],
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

feature_cols_df = pd.DataFrame({"feature": X_test.columns.tolist()})
feature_cols_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

print("saved to:", CFG.SAVE_DIR)
display(submission.head())

saved to: /kaggle/working/exp_20260409_026_realmlp_pair_te_min


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


## quick summary

In [13]:
summary_df = pd.DataFrame({
    "item": [
        "raw_cv_ba",
        "biased_cv_ba",
        "improvement",
        "n_pair_columns",
        "n_features_before_te",
        "n_features_after_te",
    ],
    "value": [
        raw_cv_ba,
        best_score,
        best_score - raw_cv_ba,
        len(TE_columns),
        len(FEATURES),
        len(X_test.columns),
    ],
})
display(summary_df)

,item,value
0,raw_cv_ba,0.978023
1,biased_cv_ba,0.978113
2,improvement,0.000090
3,n_pair_columns,9.000000
4,n_features_before_te,39.000000
5,n_features_after_te,57.000000
